# Finding Hotspots with DBSCAN and HDBSCAN

**Series:** Spatial Clustering (notebook 2 of 3)
**Time:** about 30 minutes

**In this notebook you will:**
- Understand density based clustering
- Choose DBSCAN's distance (`eps`) with a k-distance plot
- Find robbery clusters and noise points
- Let HDBSCAN choose the distance for you

**You should already know:** notebook 05a (crime points in meters).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## How DBSCAN works

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) needs two settings:

- **`eps`**: a search radius, in meters here
- **`min_samples`**: how many points must fall inside that radius

A point with at least `min_samples` neighbors within `eps` is a **core point**. Core points that
are close together join into one cluster. Points that are not near any core point are labeled
**noise** (`-1`).

Why it suits maps: clusters can be any shape (along a street, around a park), and you do not
have to say how many clusters you expect.

## Step 1: Load and prepare the data (same as notebook 05a)

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# Read the CSV straight from GitHub into memory (about 40 MB, nothing is saved to disk)
crimes = pd.read_csv("https://raw.githubusercontent.com/SpatialTurn/DataAnalysis/main/data/CrimesChicago20220225.csv")
crimes.columns = crimes.columns.str.strip()          # some column names start with a space
crimes = crimes.dropna(subset=["LATITUDE", "LONGITUDE"])

In [ ]:
# Keep three crime types and turn them into points measured in meters
crime_types = ["WEAPONS VIOLATION", "ROBBERY", "NARCOTICS"]
subset = crimes[crimes["PRIMARY DESCRIPTION"].isin(crime_types)]

points = gpd.GeoDataFrame(
    subset,
    geometry=gpd.points_from_xy(subset["LONGITUDE"], subset["LATITUDE"]),
    crs="EPSG:4326",
).to_crs(epsg=32616)      # UTM zone 16N: x and y are in meters

robbery = points[points["PRIMARY DESCRIPTION"] == "ROBBERY"].copy()
coords = np.column_stack([robbery.geometry.x, robbery.geometry.y])
print(len(robbery), "robberies")

## Step 2: Choose `eps` with a k-distance plot

For every point we measure the distance to its 50th nearest neighbor and sort those distances.
The curve bends upward where points stop being in dense areas. A good `eps` is near that bend.

In [ ]:
from sklearn.neighbors import NearestNeighbors

k = 50
distances, _ = NearestNeighbors(n_neighbors=k).fit(coords).kneighbors(coords)
k_dist = np.sort(distances[:, -1])

plt.figure(figsize=(8, 4))
plt.plot(k_dist)
plt.ylabel(f"Distance to {k}th nearest robbery (m)")
plt.xlabel("Robberies, sorted")
plt.title("k-distance plot")
plt.grid(True)
plt.show()

The bend is usually somewhere between 500 and 1,500 meters. We start with **1,000 m**.

## Step 3: Run DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=1000, min_samples=50).fit(coords)
robbery["dbscan"] = db.labels_

n_clusters = robbery["dbscan"].max() + 1
n_noise = (robbery["dbscan"] == -1).sum()
print(f"{n_clusters} clusters, {n_noise} noise points ({n_noise / len(robbery):.0%})")

## Step 4: Map the clusters

In [ ]:
def plot_clusters(gdf, label_col, title):
    fig, ax = plt.subplots(figsize=(7, 9))
    noise = gdf[gdf[label_col] == -1]
    clustered = gdf[gdf[label_col] != -1]
    noise.plot(ax=ax, color="lightgray", markersize=1, label="noise")
    clustered.plot(ax=ax, column=label_col, cmap="tab20", markersize=2, categorical=True)
    ax.set_title(title)
    ax.set_axis_off()
    plt.show()

In [ ]:
plot_clusters(robbery, "dbscan", "Robbery clusters (DBSCAN, eps = 1000 m)")

Gray points are noise. Each color is a separate cluster.

## Step 5: See how `eps` changes the result

In [ ]:
for eps in [500, 1000, 1500]:
    labels = DBSCAN(eps=eps, min_samples=50).fit(coords).labels_
    print(f"eps = {eps:>4} m: {labels.max() + 1:>3} clusters, "
          f"{(labels == -1).mean():.0%} noise")

A **small** `eps` gives many small clusters and lots of noise. A **large** `eps` merges everything
into a few big clusters. There is no single right answer; choose a value that matches your question
(a street corner? a neighborhood?).

## Step 6: HDBSCAN, which picks the distance for you

HDBSCAN tries many distances and keeps the clusters that stay stable. You only set:

- **`min_cluster_size`**: the smallest group you would call a cluster
- **`min_samples`** (optional): how strict to be about noise; larger means more noise

In [ ]:
from sklearn.cluster import HDBSCAN

hdb = HDBSCAN(min_cluster_size=50, min_samples=10).fit(coords)
robbery["hdbscan"] = hdb.labels_

print(robbery["hdbscan"].max() + 1, "clusters,",
      f"{(robbery['hdbscan'] == -1).mean():.0%} noise")
plot_clusters(robbery, "hdbscan", "Robbery clusters (HDBSCAN)")

## Try it yourself

Run DBSCAN on **narcotics** crimes instead of robbery. Do you need a different `eps`? Use the k-distance plot to decide.

<details>
<summary><b>Show a solution</b></summary>

```python
narcotics = points[points["PRIMARY DESCRIPTION"] == "NARCOTICS"].copy()
nc = np.column_stack([narcotics.geometry.x, narcotics.geometry.y])

d, _ = NearestNeighbors(n_neighbors=50).fit(nc).kneighbors(nc)
plt.plot(np.sort(d[:, -1])); plt.show()

narcotics["dbscan"] = DBSCAN(eps=800, min_samples=50).fit(nc).labels_
plot_clusters(narcotics, "dbscan", "Narcotics clusters (DBSCAN)")
```

</details>

In [ ]:
# Write your answer here

## Recap

- DBSCAN groups points that have many neighbors within a radius (`eps`).
- Points that belong to no dense area are labeled noise (`-1`).
- Use a k-distance plot to choose `eps`, and always work in meters.
- HDBSCAN removes the need to pick `eps` by testing many distances.

**Next:** [K-Means and Comparing Methods](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/05c_KMeans_and_Comparing_Methods.ipynb)